# Data Deduplication and Variable Preparation

**Master's thesis:** From Dyads to Networks: Vendor Lock-in and Systemic Vulnerability in German Public IT Procurement

**Author:** Marco Stoehr

**Notebook:** 2 of 5 (run in order; see README)

**Thesis sections:** 3.1, Appendix A.3

## Purpose
This notebook consists of two parts:

1. **Deduplication check:** investigates whether framework agreements and the call-offs drawn on them can be identified and deduplicated. The dataset does not provide sufficient information for this (see Appendix A.3), so no records are removed here. Instead, awards are coded binarily per dyad-year (award between a buyer-supplier pair in a given year: yes/no), which prevents double-counting within a year. This part documents the check reported in the thesis and does not alter the data used in later steps.
2. **Variable preparation:** transforms the cleaned award data into a dyad-year dataset (buyer-supplier-year structure) and constructs some of the variables used in the analysis.

## Input
- `data_checkpoints/data_final.csv`

## Output
- `data_analysis/dyad_years.csv`
- `data_analysis/buyer_div_seg.csv`
- `data_analysis/supplier_dom_seg.csv`



In [1]:
# import packages
import pandas as pd
import numpy as np

In [2]:
# read data
data = pd.read_csv("data_checkpoints/data_final.csv")

### Part I: check for deduplication 

In [3]:
# Full column inventory
print(f"{len(data.columns)} columns:")
print(sorted(data.columns.tolist()))

51 columns:
['AWARD_VALUE_EURO', 'B_CONTRACTOR_SME', 'B_EU_FUNDS', 'B_FRA_AGREEMENT', 'B_FRA_CONTRACT', 'B_MULTIPLE_CAE', 'B_ON_BEHALF', 'B_SUBCONTRACTED', 'CAE_NAME', 'CAE_NAME_CANONICAL', 'CAE_NAME_NORM', 'CAE_NAME_NORM_SORTED', 'CAE_NATIONALID', 'CAE_TYPE', 'CPV', 'CRIT_CODE', 'DT_AWARD', 'FRA_ESTIMATED', 'ID_NOTICE_CAN', 'INFO_ON_NON_AWARD', 'INFO_UNPUBLISHED', 'ISO_COUNTRY_CODE', 'IS_SPLIT', 'IS_SPLIT_BUYER', 'LOTS_NUMBER', 'MAIN_ACTIVITY', 'NUMBER_AWARDS', 'NUMBER_OFFERS', 'TAL_LOCATION_NUTS', 'TOP_TYPE', 'TYPE_OF_CONTRACT', 'Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2', 'Unnamed: 0.3', 'Unnamed: 0.4', 'Unnamed: 0.5', 'Unnamed: 0.6', 'Unnamed: 0.7', 'VALUE_EURO', 'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2', 'WIN_COUNTRY_CODE', 'WIN_NAME', 'WIN_NAME_CANONICAL', 'WIN_NAME_FINAL', 'WIN_NAME_NORM', 'WIN_NAME_NORM_SORTED', 'WIN_NATIONALID', 'YEAR', 'YEAR_ID']


In [4]:
# 1. Identify framework agreement related columns
fra_cols = [col for col in data.columns if any(x in col.upper() for x in ['FRA', 'FRAMEWORK', 'RAHMEN'])]
print("=== Framework-related columns ===")
for col in fra_cols:
    print(f"\n--- {col} ---")
    print(f"dtype: {data[col].dtype}")
    print(f"non-null: {data[col].notna().sum()} / {len(data)}")
    print(f"unique values: {data[col].nunique()}")
    print(data[col].value_counts(dropna=False).head(10))

# 2. Check for reference/ID fields that might link call-offs to parent agreements
id_cols = [col for col in data.columns if any(x in col.upper() for x in ['REF', 'PARENT', 'ID_NOTICE', 'NOTICE_NUMBER', 'DOC_NUMBER', 'FUTURE_NOTICE', 'RELATES'])]
print("\n\n=== Potential linking columns ===")
for col in id_cols:
    print(f"\n--- {col} ---")
    print(f"dtype: {data[col].dtype}")
    print(f"non-null: {data[col].notna().sum()} / {len(data)}")
    print(f"unique values: {data[col].nunique()}")
    print(data[col].value_counts(dropna=False).head(5))

=== Framework-related columns ===

--- B_FRA_AGREEMENT ---
dtype: str
non-null: 56242 / 56242
unique values: 2
B_FRA_AGREEMENT
N    30398
Y    25844
Name: count, dtype: int64

--- FRA_ESTIMATED ---
dtype: str
non-null: 10020 / 56242
unique values: 7
FRA_ESTIMATED
NaN    46222
K       5455
A       3108
KA      1045
C        268
KC        98
AC        41
KAC        5
Name: count, dtype: int64

--- B_FRA_CONTRACT ---
dtype: str
non-null: 56242 / 56242
unique values: 2
B_FRA_CONTRACT
N    34217
Y    22025
Name: count, dtype: int64


=== Potential linking columns ===

--- ID_NOTICE_CAN ---
dtype: int64
non-null: 56242 / 56242
unique values: 31191
ID_NOTICE_CAN
2022169652    621
2023216865    570
2023186841    400
2022427692    324
2023188355    324
Name: count, dtype: int64


In [5]:
# 1. Cross-tab: Agreement x Contract
print("=== Cross-tab B_FRA_AGREEMENT x B_FRA_CONTRACT ===")
ct = pd.crosstab(data['B_FRA_AGREEMENT'], data['B_FRA_CONTRACT'], margins=True)
print(ct)

# 2. FRA_ESTIMATED breakdown for those NOT marked as framework
print("\n=== FRA_ESTIMATED where B_FRA_AGREEMENT = N ===")
non_fra = data[data['B_FRA_AGREEMENT'] == 'N']
print(non_fra['FRA_ESTIMATED'].value_counts(dropna=False))

# 3. The critical overlap: marked as BOTH agreement AND call-off?
both = data[(data['B_FRA_AGREEMENT'] == 'Y') & (data['B_FRA_CONTRACT'] == 'Y')]
print(f"\n=== Both Agreement AND Contract: {len(both)} rows ===")

# 4. Key question: can call-offs be linked to framework agreements?
# Check if there's any field that references the parent notice
link_cols = [col for col in data.columns if any(x in col.upper() for x in 
    ['PARENT', 'PREVIOUS', 'RELATES', 'REF_NOTICE', 'ID_NOTICE_CN', 'FUTURE'])]
print(f"\n=== Potential parent-linking columns ===")
for col in link_cols:
    print(f"\n--- {col} ---")
    print(f"non-null: {data[col].notna().sum()} / {len(data)}")
    print(data[col].value_counts(dropna=False).head(5))

=== Cross-tab B_FRA_AGREEMENT x B_FRA_CONTRACT ===


B_FRA_CONTRACT       N      Y    All
B_FRA_AGREEMENT                     
N                28945   1453  30398
Y                 5272  20572  25844
All              34217  22025  56242

=== FRA_ESTIMATED where B_FRA_AGREEMENT = N ===
FRA_ESTIMATED
NaN    29402
K        492
A        222
C        200
KC        47
AC        20
KA        10
KAC        5
Name: count, dtype: int64

=== Both Agreement AND Contract: 20572 rows ===

=== Potential parent-linking columns ===


In [6]:
# 1. For the "both Y" call-offs: how many per buyer-supplier-year?
calloffs = data[(data['B_FRA_AGREEMENT'] == 'Y') & (data['B_FRA_CONTRACT'] == 'Y')]
regular = data[(data['B_FRA_AGREEMENT'] == 'N') & (data['B_FRA_CONTRACT'] == 'N')]

print("=== Call-offs (both Y): buyer-supplier-year concentration ===")
co_groups = calloffs.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']).size()
print(f"Unique buyer-supplier-year combos: {len(co_groups)}")
print(f"Mean contracts per combo: {co_groups.mean():.2f}")
print(f"Median: {co_groups.median():.0f}")
print(f"Max: {co_groups.max()}")
print(co_groups.value_counts().sort_index().head(10))

print("\n=== Regular contracts (both N): same breakdown ===")
reg_groups = regular.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']).size()
print(f"Unique buyer-supplier-year combos: {len(reg_groups)}")
print(f"Mean contracts per combo: {reg_groups.mean():.2f}")
print(f"Median: {reg_groups.median():.0f}")
print(f"Max: {reg_groups.max()}")
print(reg_groups.value_counts().sort_index().head(10))

# 2. Framework establishments: do they precede call-offs for same buyer-supplier?
establishments = data[(data['B_FRA_AGREEMENT'] == 'Y') & (data['B_FRA_CONTRACT'] == 'N')]
print(f"\n=== Framework establishments (FRA=Y, CON=N): {len(establishments)} ===")
print(f"Unique buyer-supplier pairs: {establishments.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL']).ngroups}")

# 3. The 1,456 ambiguous cases
ambiguous = data[(data['B_FRA_AGREEMENT'] == 'N') & (data['B_FRA_CONTRACT'] == 'Y')]
print(f"\n=== Ambiguous (FRA=N, CON=Y): {len(ambiguous)} ===")
print(f"FRA_ESTIMATED distribution:")
print(ambiguous['FRA_ESTIMATED'].value_counts(dropna=False))

=== Call-offs (both Y): buyer-supplier-year concentration ===
Unique buyer-supplier-year combos: 12692
Mean contracts per combo: 1.62
Median: 1
Max: 120
1     9466
2     1827
3      639
4      295
5      149
6      127
7       66
8       31
9       20
10      14
Name: count, dtype: int64

=== Regular contracts (both N): same breakdown ===
Unique buyer-supplier-year combos: 22001
Mean contracts per combo: 1.32
Median: 1
Max: 306
1     18906
2      1994
3       530
4       240
5        92
6        75
7        40
8        27
9        17
10       14
Name: count, dtype: int64

=== Framework establishments (FRA=Y, CON=N): 5272 ===
Unique buyer-supplier pairs: 3053

=== Ambiguous (FRA=N, CON=Y): 1453 ===
FRA_ESTIMATED distribution:
FRA_ESTIMATED
NaN    931
C      199
K      188
A       64
KC      36
AC      20
KA      10
KAC      5
Name: count, dtype: int64


In [7]:
# 1. Framework establishments without any call-off from same buyer-supplier
estab = data[(data['B_FRA_AGREEMENT'] == 'Y') & (data['B_FRA_CONTRACT'] == 'N')]
calloff = data[(data['B_FRA_AGREEMENT'] == 'Y') & (data['B_FRA_CONTRACT'] == 'Y')]

estab_pairs = set(zip(estab['CAE_NAME_CANONICAL'], estab['WIN_NAME_FINAL']))
calloff_pairs = set(zip(calloff['CAE_NAME_CANONICAL'], calloff['WIN_NAME_FINAL']))

estab_only = estab_pairs - calloff_pairs
estab_with_calloff = estab_pairs & calloff_pairs

print(f"=== Framework establishments: pair analysis ===")
print(f"Total unique buyer-supplier pairs with establishment: {len(estab_pairs)}")
print(f"Pairs that also have call-offs: {len(estab_with_calloff)}")
print(f"Pairs with establishment but NO call-off ever: {len(estab_only)}")

# 2. What does the proposed dyad-year collapsing actually look like?
print("\n=== Impact of dyad-year collapsing ===")
all_dyad_years = data.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']).size()
print(f"Total rows: {len(data)}")
print(f"Unique dyad-years: {len(all_dyad_years)}")
print(f"Rows that would collapse: {len(data) - len(all_dyad_years)} ({(len(data) - len(all_dyad_years))/len(data)*100:.1f}%)")

# 3. Impact on buyer contract counts (threshold-relevant!)
current_counts = data.groupby('CAE_NAME_CANONICAL')['WIN_NAME_FINAL'].count()
collapsed_counts = data.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']).ngroup()
collapsed_buyer_counts = data.drop_duplicates(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']).groupby('CAE_NAME_CANONICAL').size()

print("\n=== Threshold impact ===")
print(f"Buyers ≥5 contracts (current): {(current_counts >= 5).sum()}")
print(f"Buyers ≥5 dyad-years (collapsed): {(collapsed_buyer_counts >= 5).sum()}")
print(f"Buyers losing ≥5 status: {(current_counts >= 5).sum() - (collapsed_buyer_counts >= 5).sum()}")

=== Framework establishments: pair analysis ===
Total unique buyer-supplier pairs with establishment: 3053
Pairs that also have call-offs: 690
Pairs with establishment but NO call-off ever: 2363

=== Impact of dyad-year collapsing ===
Total rows: 56242
Unique dyad-years: 38023
Rows that would collapse: 18219 (32.4%)

=== Threshold impact ===
Buyers ≥5 contracts (current): 1766
Buyers ≥5 dyad-years (collapsed): 1496
Buyers losing ≥5 status: 270


In [8]:
# --- Consequence of the binary dyad-year coding -----------------------------
# Diagnostic only: `collapsed` is not carried into Part 2. It quantifies how
# much of the row count is absorbed by the binary coding and what this implies
# for buyer-level sample thresholds.

DYAD_KEY = ['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']

# The collapse key must be complete, otherwise dyad-year counts silently
# drop rows.
print("Missing values in collapse key:")
print(data[DYAD_KEY].isna().sum())

collapsed = data.drop_duplicates(DYAD_KEY)

n_rows, n_dyad_years = len(data), len(collapsed)
n_absorbed = n_rows - n_dyad_years

print("\n=== Binary dyad-year coding: effect on row count ===")
print(f"Rows: {n_rows} -> {n_dyad_years} "
      f"(-{n_absorbed}, {n_absorbed / n_rows * 100:.1f}%)")

# --- Buyer-level distribution ----------------------------------------------
buyer_dyad_years = collapsed.groupby('CAE_NAME_CANONICAL').size()

print("\n=== Buyers by number of dyad-years ===")
print(f"Median: {buyer_dyad_years.median():.0f}, "
      f"P75: {buyer_dyad_years.quantile(.75):.0f}, "
      f"Max: {buyer_dyad_years.max()}")

# --- Effect on a buyer-level sample threshold -------------------------------
# The binary coding changes which buyers pass a minimum-activity threshold,
# because repeated awards to the same supplier within a year no longer count
# separately. This is the substantive cost of the approach and is reported
# in the methods section.
MIN_OBS = 5

buyer_rows = data.groupby('CAE_NAME_CANONICAL').size()
passing_rows = (buyer_rows >= MIN_OBS).sum()
passing_dyads = (buyer_dyad_years >= MIN_OBS).sum()

covered = collapsed['CAE_NAME_CANONICAL'].isin(
    buyer_dyad_years[buyer_dyad_years >= MIN_OBS].index
).sum()

print(f"\n=== Threshold effect (>= {MIN_OBS}) ===")
print(f"Buyers passing on raw award rows:  {passing_rows}")
print(f"Buyers passing on dyad-years:      {passing_dyads} "
      f"(-{passing_rows - passing_dyads})")
print(f"Dyad-years covered by these buyers: {covered} "
      f"({covered / n_dyad_years * 100:.1f}%)")

Missing values in collapse key:
CAE_NAME_CANONICAL    0
WIN_NAME_FINAL        0
YEAR                  0
dtype: int64

=== Binary dyad-year coding: effect on row count ===
Rows: 56242 -> 38023 (-18219, 32.4%)

=== Buyers by number of dyad-years ===
Median: 2, P75: 5, Max: 639

=== Threshold effect (>= 5) ===
Buyers passing on raw award rows:  1766
Buyers passing on dyad-years:      1496 (-270)
Dyad-years covered by these buyers: 31230 (82.1%)


In [9]:
# Can we identify "call-off only" dyad-years?
# These are dyad-years where ALL rows are B_FRA_CONTRACT=Y and NONE are establishments

dyad_info = data.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']).agg(
    n_rows=('B_FRA_CONTRACT', 'size'),
    n_calloff=('B_FRA_CONTRACT', lambda x: (x == 'Y').sum()),
    n_establishment=('B_FRA_AGREEMENT', lambda x: ((x == 'Y') & (data.loc[x.index, 'B_FRA_CONTRACT'] == 'N')).sum()),
    n_regular=('B_FRA_AGREEMENT', lambda x: ((x == 'N') & (data.loc[x.index, 'B_FRA_CONTRACT'] == 'N')).sum())
).reset_index()

# Classify dyad-years
dyad_info['type'] = 'mixed'
dyad_info.loc[dyad_info['n_calloff'] == dyad_info['n_rows'], 'type'] = 'calloff_only'
dyad_info.loc[dyad_info['n_regular'] == dyad_info['n_rows'], 'type'] = 'regular_only'
dyad_info.loc[dyad_info['n_establishment'] == dyad_info['n_rows'], 'type'] = 'establishment_only'

print("=== Dyad-year classification ===")
print(dyad_info['type'].value_counts())

# For call-off-only dyad-years: is there a prior establishment from the same pair?
calloff_only = dyad_info[dyad_info['type'] == 'calloff_only']
estab_pairs_years = data[(data['B_FRA_AGREEMENT'] == 'Y') & (data['B_FRA_CONTRACT'] == 'N')][
    ['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']
].rename(columns={'YEAR': 'estab_year'})

# Merge to find if a prior establishment exists
calloff_with_estab = calloff_only.merge(
    estab_pairs_years, on=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL'], how='left'
)
# Same-year call-offs are the normal case (framework signed and used within
# one year), so the origin year must be included, not excluded.
calloff_with_estab['has_prior_estab'] = (
    calloff_with_estab['estab_year'] <= calloff_with_estab['YEAR']
)
prior_estab_exists = calloff_with_estab.groupby(
    ['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']
)['has_prior_estab'].any()

print(f"\n=== Call-off-only dyad-years with identifiable prior establishment ===")
print(f"Total call-off-only dyad-years: {len(calloff_only)}")
print(f"With prior establishment from same pair: {prior_estab_exists.sum()}")
print(f"Without (can't trace origin): {(~prior_estab_exists).sum()}")

=== Dyad-year classification ===
type
regular_only          21404
calloff_only          12709
establishment_only     3072
mixed                   838
Name: count, dtype: int64

=== Call-off-only dyad-years with identifiable prior establishment ===
Total call-off-only dyad-years: 12709
With prior establishment from same pair: 713
Without (can't trace origin): 11996


### Conclusion: no deduplication of framework agreements

TED provides no field linking a call-off to its parent framework agreement.
`ID_NOTICE_CAN` identifies the notice, not the award, and no reference to a
preceding notice exists in the data. The two framework flags do not partition
the records either: 20,572 rows (36.6%) are marked simultaneously as
establishing a framework agreement and as a call-off within one, because
`B_FRA_CONTRACT` is an inferred rather than a reported variable.

Nor can the link be reconstructed indirectly. Of 12,709 call-off dyad-years,
only 713 (5.6%) can be matched to any earlier framework agreement of the same
buyer-supplier pair, and 2,363 of 3,053 framework agreements show no call-off
at all; call-offs generally require no separate publication, so the observed
ones are a non-random subset shaped by reporting practice. Deduplication would
therefore penalise the more transparently reporting buyers.

The analysis instead codes each buyer-supplier pair binarily per year. This
absorbs 18,219 of 56,242 rows (32.4%), leaving 38,023 dyad-years, and is
invariant to the framework/call-off ambiguity by construction.

## Part II: variable preparation

Before variable preparation, EU-Institutions were removed from the CAE_NAME fields since those institutions are neither genuin German institutions nor do they fall under the German procurement law.

In [10]:
# Which buyers are classified as EU institutions / international organisations?
print(data.loc[data['CAE_TYPE'].isin(['5', '5A']), 'CAE_NAME_CANONICAL']
      .value_counts())

CAE_NAME_CANONICAL
European Central Bank                                                                                                                                            141
European Aviation Safety Agency                                                                                                                                   52
European Union Aviation Safety Agency                                                                                                                             47
European Insurance and Occupational Pensions Authority                                                                                                            28
EIOPA                                                                                                                                                              8
European Commission; Joint Research Centre; Institute for Transuranium Elements                                                                             

In [11]:
# --- Analysis population ---------------------------------------------------
# CAE_TYPE 5 and 5A identify EU
# institutions and international organisations, which appear in the German
# selection because of their location rather than because they are German
# public buyers.
n_before = len(data)
data = data[~data['CAE_TYPE'].isin(['5', '5A'])].copy()

print(f"Analysis population: {n_before} -> {len(data)} rows "
      f"(-{n_before - len(data)})")
print(f"Buyers: {data['CAE_NAME_CANONICAL'].nunique()}")

Analysis population: 56242 -> 55940 rows (-302)
Buyers: 5591


In [12]:
# 1. Collapsed dyad-year dataset with tie intensity
dyad_years = (
    data.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'])
    .agg(
        n_contracts_ijt=('YEAR', 'size'),  # rows behind this dyad-year
        cpv_segments=('CPV', lambda x: set(x.astype(str).str[:2]))  # which CPV segments
    )
    .reset_index()
)

# 2. Sort for cumulative calculations
dyad_years = dyad_years.sort_values(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'])

# 3. Cumulative dyad-years before year t (excluding current year)
dyad_years['cumulative_ties_ijt'] = (
    dyad_years.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL'])
    .cumcount()  # 0-indexed, so first appearance = 0 = no prior ties
)

# 4. ever_tied_ijt: was there any tie before this year?
dyad_years['ever_tied_ijt'] = (dyad_years['cumulative_ties_ijt'] > 0).astype(int)

# 5. years_since_last_ijt
dyad_years['prev_year'] = (
    dyad_years.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL'])['YEAR']
    .shift(1)
)
dyad_years['years_since_last_ijt'] = dyad_years['YEAR'] - dyad_years['prev_year']

# 5b. A missing value here means "no prior tie", not "unobserved". Filling with
# 0 and flagging first appearances keeps all rows in the estimation and gives
# the no-prior-tie case its own coefficient instead of dropping it.
dyad_years['first_tie_ijt'] = dyad_years['years_since_last_ijt'].isna().astype(int)
dyad_years['ysl_filled_ijt'] = dyad_years['years_since_last_ijt'].fillna(0)


# 6. first_observed_year per buyer (left-censoring control)
dyad_years['first_observed_year_buyer'] = (
    dyad_years.groupby('CAE_NAME_CANONICAL')['YEAR']
    .transform('min')
)

# 7. Quick validation
print(f"=== Dyad-year dataset ===")
print(f"Rows: {len(dyad_years)}")
print(f"Unique buyers: {dyad_years['CAE_NAME_CANONICAL'].nunique()}")
print(f"Unique suppliers: {dyad_years['WIN_NAME_FINAL'].nunique()}")

print(f"\n=== ever_tied_ijt ===")
print(dyad_years['ever_tied_ijt'].value_counts())

print(f"\n=== cumulative_ties_ijt ===")
print(dyad_years['cumulative_ties_ijt'].describe())

print(f"\n=== years_since_last_ijt (where applicable) ===")
print(dyad_years['years_since_last_ijt'].describe())

print(f"\n=== n_contracts_ijt ===")
print(dyad_years['n_contracts_ijt'].describe())

print(f"\n=== first_observed_year_buyer ===")
print(dyad_years['first_observed_year_buyer'].value_counts().sort_index())

=== Dyad-year dataset ===
Rows: 37775
Unique buyers: 5591
Unique suppliers: 10440

=== ever_tied_ijt ===
ever_tied_ijt
0    29888
1     7887
Name: count, dtype: int64

=== cumulative_ties_ijt ===
count    37775.000000
mean         0.415433
std          1.108834
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         14.000000
Name: cumulative_ties_ijt, dtype: float64

=== years_since_last_ijt (where applicable) ===
count    7887.000000
mean        2.430328
std         1.883881
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        14.000000
Name: years_since_last_ijt, dtype: float64

=== n_contracts_ijt ===
count    37775.000000
mean         1.480874
std          4.555326
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        560.000000
Name: n_contracts_ijt, dtype: float64

=== first_observed_year_buyer ===
first_observed_year_buyer
2009    11977
2010     320

In [13]:
# --- 1. Buyer diversification (buyer-year level) ---

buyer_div = (
    dyad_years.groupby(['CAE_NAME_CANONICAL', 'YEAR'])
    .agg(n_suppliers_it=('WIN_NAME_FINAL', 'nunique'))
    .reset_index()
)
buyer_div['log_n_suppliers_it'] = np.log1p(buyer_div['n_suppliers_it'])

print("=== Buyer diversification (aggregate) ===")
print(f"Rows: {len(buyer_div)}")
print(buyer_div['n_suppliers_it'].describe().round(3))

# --- 2. Supplier market dominance (supplier-year level) ---

n_active_buyers = dyad_years.groupby('YEAR')['CAE_NAME_CANONICAL'].nunique().rename('n_active_buyers_t')

supplier_dom = (
    dyad_years.groupby(['WIN_NAME_FINAL', 'YEAR'])
    .agg(n_buyers_jt=('CAE_NAME_CANONICAL', 'nunique'))
    .reset_index()
    .merge(n_active_buyers, on='YEAR')
)
supplier_dom['supplier_share_jt'] = supplier_dom['n_buyers_jt'] / supplier_dom['n_active_buyers_t']

print("\n=== Supplier market dominance (aggregate) ===")
print(f"Rows: {len(supplier_dom)}")
print(supplier_dom[['n_buyers_jt', 'supplier_share_jt']].describe().round(4))
print("\nTop 10 supplier-years by market share:")
print(supplier_dom.nlargest(10, 'supplier_share_jt')[['WIN_NAME_FINAL', 'YEAR', 'n_buyers_jt', 'supplier_share_jt']].to_string(index=False))

# --- 3. Segment-specific versions ---

dyad_cpv = dyad_years.explode('cpv_segments').rename(columns={'cpv_segments': 'cpv2'})

# Buyer diversification per segment
buyer_div_seg = (
    dyad_cpv.groupby(['CAE_NAME_CANONICAL', 'YEAR', 'cpv2'])
    .agg(n_suppliers_it_seg=('WIN_NAME_FINAL', 'nunique'))
    .reset_index()
)
buyer_div_seg['log_n_suppliers_it_seg'] = np.log1p(buyer_div_seg['n_suppliers_it_seg'])

# Supplier dominance per segment
n_active_buyers_seg = dyad_cpv.groupby(['YEAR', 'cpv2'])['CAE_NAME_CANONICAL'].nunique().reset_index(name='n_active_buyers_seg_t')

supplier_dom_seg = (
    dyad_cpv.groupby(['WIN_NAME_FINAL', 'YEAR', 'cpv2'])
    .agg(n_buyers_jt_seg=('CAE_NAME_CANONICAL', 'nunique'))
    .reset_index()
    .merge(n_active_buyers_seg, on=['YEAR', 'cpv2'])
)
supplier_dom_seg['supplier_share_jt_seg'] = supplier_dom_seg['n_buyers_jt_seg'] / supplier_dom_seg['n_active_buyers_seg_t']

print("\n\n=== Segment-specific buyer diversification ===")
for seg in ['30', '48', '72']:
    sub = buyer_div_seg[buyer_div_seg['cpv2'] == seg]
    print(f"\nCPV {seg} (n={len(sub)}): mean={sub['n_suppliers_it_seg'].mean():.2f}, median={sub['n_suppliers_it_seg'].median():.0f}")

print("\n=== Segment-specific supplier dominance (top 3 per segment) ===")
for seg in ['30', '48', '72']:
    sub = supplier_dom_seg[supplier_dom_seg['cpv2'] == seg]
    print(f"\n--- CPV {seg} ---")
    print(sub.nlargest(3, 'supplier_share_jt_seg')[['WIN_NAME_FINAL', 'YEAR', 'n_buyers_jt_seg', 'supplier_share_jt_seg']].to_string(index=False))

=== Buyer diversification (aggregate) ===
Rows: 13651
count    13651.000
mean         2.767
std          4.786
min          1.000
25%          1.000
50%          1.000
75%          3.000
max        156.000
Name: n_suppliers_it, dtype: float64

=== Supplier market dominance (aggregate) ===
Rows: 19494
       n_buyers_jt  supplier_share_jt
count   19494.0000         19494.0000
mean        1.9378             0.0020
std         5.0453             0.0049
min         1.0000             0.0006
25%         1.0000             0.0007
50%         1.0000             0.0011
75%         1.0000             0.0018
max       198.0000             0.1401

Top 10 supplier-years by market share:
        WIN_NAME_FINAL  YEAR  n_buyers_jt  supplier_share_jt
            Bechtle AG  2017          124           0.140113
            Bechtle AG  2022          198           0.127660
            Bechtle AG  2021          193           0.125979
            Bechtle AG  2020          169           0.122909
Ricoh Deuts

In [14]:
# --- Merge aggregate controls onto the dyad panel --------------------------
# Idempotent: drop previously merged columns first, so re-running this cell
# cannot create suffixed duplicates.

MERGED_COLS = [
    'n_suppliers_it', 'log_n_suppliers_it',
    'n_buyers_jt', 'n_active_buyers_t', 'supplier_share_jt',
    'n_other_suppliers_it', 'log_n_other_suppliers_it',
    'n_other_buyers_jt', 'supplier_share_excl_i_jt',
]

dyad_years = (
    dyad_years.drop(columns=MERGED_COLS, errors='ignore')
    .merge(buyer_div, on=['CAE_NAME_CANONICAL', 'YEAR'], how='left')
    .merge(supplier_dom, on=['WIN_NAME_FINAL', 'YEAR'], how='left')
)

# --- Leave-one-out controls ------------------------------------------------
# The focal dyad must not enter its own controls: a buyer-year's supplier count
# contains the focal supplier, and a supplier-year's buyer count contains the
# focal buyer. Each dyad appears at most once per year, so subtracting one is
# exact rather than approximate. The aggregate versions above are kept for
# descriptive use.

dyad_years['n_other_suppliers_it'] = dyad_years['n_suppliers_it'] - 1
dyad_years['log_n_other_suppliers_it'] = np.log1p(dyad_years['n_other_suppliers_it'])

dyad_years['n_other_buyers_jt'] = dyad_years['n_buyers_jt'] - 1
dyad_years['supplier_share_excl_i_jt'] = (
    dyad_years['n_other_buyers_jt'] / (dyad_years['n_active_buyers_t'] - 1)
)

print("=== Consolidated dyad_years ===")
print(f"Rows: {len(dyad_years)}")
print(f"Columns: {list(dyad_years.columns)}")
print("\nNull check:")
print(dyad_years.isnull().sum())
print("\nLeave-one-out controls:")
print(dyad_years[['n_other_suppliers_it', 'n_other_buyers_jt',
                  'supplier_share_excl_i_jt']].describe().round(4))

=== Consolidated dyad_years ===
Rows: 37775
Columns: ['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR', 'n_contracts_ijt', 'cpv_segments', 'cumulative_ties_ijt', 'ever_tied_ijt', 'prev_year', 'years_since_last_ijt', 'first_tie_ijt', 'ysl_filled_ijt', 'first_observed_year_buyer', 'n_suppliers_it', 'log_n_suppliers_it', 'n_buyers_jt', 'n_active_buyers_t', 'supplier_share_jt', 'n_other_suppliers_it', 'log_n_other_suppliers_it', 'n_other_buyers_jt', 'supplier_share_excl_i_jt']

Null check:
CAE_NAME_CANONICAL               0
WIN_NAME_FINAL                   0
YEAR                             0
n_contracts_ijt                  0
cpv_segments                     0
cumulative_ties_ijt              0
ever_tied_ijt                    0
prev_year                    29888
years_since_last_ijt         29888
first_tie_ijt                    0
ysl_filled_ijt                   0
first_observed_year_buyer        0
n_suppliers_it                   0
log_n_suppliers_it               0
n_buyers_jt          

In [15]:
# Convert sets to strings for CSV storage
dyad_years['cpv_segments_str'] = dyad_years['cpv_segments'].apply(lambda x: '|'.join(sorted(x)))

# Save all three
dyad_years.to_csv('data_analysis/dyad_years.csv', index=False)
buyer_div_seg.to_csv('data_analysis/buyer_div_seg.csv', index=False)
supplier_dom_seg.to_csv('data_analysis/supplier_dom_seg.csv', index=False)

print("Saved:")
print(f"  dyad_years.csv ({len(dyad_years)} rows, {len(dyad_years.columns)} cols)")
print(f"  buyer_div_seg.csv ({len(buyer_div_seg)} rows)")
print(f"  supplier_dom_seg.csv ({len(supplier_dom_seg)} rows)")

Saved:
  dyad_years.csv (37775 rows, 22 cols)
  buyer_div_seg.csv (17744 rows)
  supplier_dom_seg.csv (21391 rows)


In [16]:
# last NA-check
print(f"=== NA check on all columns in raw data ({len(data.columns)} cols) ===\n")
for col in data.columns:
    na_count = data[col].isna().sum()
    na_pct = data[col].isna().mean() * 100
    print(f"{col}: {na_count} NA ({na_pct:.1f}%)")

=== NA check on all columns in raw data (51 cols) ===

Unnamed: 0.7: 0 NA (0.0%)
Unnamed: 0.6: 0 NA (0.0%)
Unnamed: 0.5: 0 NA (0.0%)
Unnamed: 0.4: 0 NA (0.0%)
Unnamed: 0.3: 0 NA (0.0%)
Unnamed: 0.2: 0 NA (0.0%)
Unnamed: 0.1: 0 NA (0.0%)
Unnamed: 0: 0 NA (0.0%)
ID_NOTICE_CAN: 0 NA (0.0%)
YEAR: 0 NA (0.0%)
CAE_NAME: 0 NA (0.0%)
ISO_COUNTRY_CODE: 0 NA (0.0%)
CAE_TYPE: 0 NA (0.0%)
MAIN_ACTIVITY: 185 NA (0.3%)
B_ON_BEHALF: 881 NA (1.6%)
TYPE_OF_CONTRACT: 0 NA (0.0%)
TAL_LOCATION_NUTS: 5622 NA (10.1%)
B_FRA_AGREEMENT: 0 NA (0.0%)
FRA_ESTIMATED: 46039 NA (82.3%)
B_FRA_CONTRACT: 0 NA (0.0%)
CPV: 0 NA (0.0%)
LOTS_NUMBER: 804 NA (1.4%)
VALUE_EURO: 10045 NA (18.0%)
VALUE_EURO_FIN_1: 8335 NA (14.9%)
VALUE_EURO_FIN_2: 8335 NA (14.9%)
B_EU_FUNDS: 2257 NA (4.0%)
TOP_TYPE: 157 NA (0.3%)
CRIT_CODE: 3060 NA (5.5%)
NUMBER_AWARDS: 0 NA (0.0%)
INFO_ON_NON_AWARD: 55940 NA (100.0%)
INFO_UNPUBLISHED: 0 NA (0.0%)
WIN_NAME: 0 NA (0.0%)
WIN_NATIONALID: 55696 NA (99.6%)
WIN_COUNTRY_CODE: 695 NA (1.2%)
B_CONTRACTO

In [17]:
# Clean up unnamed columns
unnamed = [col for col in data.columns if col.startswith('Unnamed')]
print(f"Dropping {len(unnamed)} unnamed columns: {unnamed}")
data = data.drop(columns=unnamed)

# CAE_TYPE: can we recover missings from other observations of same buyer?
cae_type_known = data.dropna(subset=['CAE_TYPE']).drop_duplicates('CAE_NAME_CANONICAL')[['CAE_NAME_CANONICAL', 'CAE_TYPE']]
print(f"\n=== CAE_TYPE recovery ===")
print(f"Buyers with known CAE_TYPE: {len(cae_type_known)} / {data['CAE_NAME_CANONICAL'].nunique()}")

# How many of the NA rows can we fill from the same buyer's other entries?
na_rows = data[data['CAE_TYPE'].isna()]
recoverable = na_rows['CAE_NAME_CANONICAL'].isin(cae_type_known['CAE_NAME_CANONICAL']).sum()
print(f"NA rows: {len(na_rows)}, recoverable from same buyer: {recoverable}")

# NUMBER_OFFERS: distribution for non-missing
print(f"\n=== NUMBER_OFFERS ===")
print(data['NUMBER_OFFERS'].describe().round(2))
print(f"Rows with 1 offer (single bid): {(data['NUMBER_OFFERS'] == 1).sum()} ({(data['NUMBER_OFFERS'] == 1).mean()*100:.1f}%)")

Dropping 8 unnamed columns: ['Unnamed: 0.7', 'Unnamed: 0.6', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0']

=== CAE_TYPE recovery ===
Buyers with known CAE_TYPE: 5591 / 5591
NA rows: 0, recoverable from same buyer: 0

=== NUMBER_OFFERS ===
count    52488.00
mean         5.00
std         17.91
min          0.00
25%          1.00
50%          3.00
75%          5.00
max        999.00
Name: NUMBER_OFFERS, dtype: float64
Rows with 1 offer (single bid): 15252 (27.3%)


In [18]:
# NUMBER_OFFERS: a contract award notice cannot have zero tenders, and 999
# looks like a filler value. Inspect the tails before using this variable.
print("\n=== NUMBER_OFFERS ===")
print(data['NUMBER_OFFERS'].describe().round(2))
print(f"\nZero offers: {(data['NUMBER_OFFERS'] == 0).sum()}")
print(f"Offers >= 100: {(data['NUMBER_OFFERS'] >= 100).sum()}")
print("\nMost frequent values >= 50:")
print(data.loc[data['NUMBER_OFFERS'] >= 50, 'NUMBER_OFFERS']
      .value_counts().head(10))
print(f"\nSingle bid: {(data['NUMBER_OFFERS'] == 1).sum()} "
      f"({(data['NUMBER_OFFERS'] == 1).mean()*100:.1f}%)")


=== NUMBER_OFFERS ===
count    52488.00
mean         5.00
std         17.91
min          0.00
25%          1.00
50%          3.00
75%          5.00
max        999.00
Name: NUMBER_OFFERS, dtype: float64

Zero offers: 40
Offers >= 100: 365

Most frequent values >= 50:
NUMBER_OFFERS
99.0     98
107.0    92
101.0    88
105.0    88
100.0    86
94.0     82
61.0     41
999.0    11
58.0      1
77.0      1
Name: count, dtype: int64

Single bid: 15252 (27.3%)


In [19]:
# --- Single-bid indicator --------------------------------------------------
# A contract award notice cannot have zero tenders, so non-positive values are
# recording errors rather than genuine observations and are set to missing
# instead of being counted as competitive.
data.loc[data['NUMBER_OFFERS'] <= 0, 'NUMBER_OFFERS'] = np.nan

data['single_bid'] = (data['NUMBER_OFFERS'] == 1).astype(float)
data.loc[data['NUMBER_OFFERS'].isna(), 'single_bid'] = np.nan

# --- Aggregate to dyad-year level ------------------------------------------
# max() ignores missing values and returns NaN only if all rows are missing,
# so a dyad-year counts as single-bid if at least one of its awards was.
dyad_attrs = (
    data.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'])
    .agg(cae_type_ijt=('CAE_TYPE', 'first'),
         has_single_bid_ijt=('single_bid', 'max'))
    .reset_index()
)

# --- Merge (idempotent) ----------------------------------------------------
dyad_years = (
    dyad_years.drop(columns=['cae_type_ijt', 'has_single_bid_ijt'], errors='ignore')
    .merge(dyad_attrs, on=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'], how='left')
)

print("=== Updated dyad_years ===")
print(f"Rows: {len(dyad_years)}")
print("\ncae_type_ijt:")
print(dyad_years['cae_type_ijt'].value_counts(dropna=False))
print("\nhas_single_bid_ijt:")
print(dyad_years['has_single_bid_ijt'].value_counts(dropna=False))


=== Updated dyad_years ===
Rows: 37775

cae_type_ijt:
cae_type_ijt
6    11942
3    10562
8     9260
1     3877
4     1130
N      656
R      279
Z       69
Name: count, dtype: int64

has_single_bid_ijt:
has_single_bid_ijt
0.0    24121
1.0    11094
NaN     2560
Name: count, dtype: int64


In [20]:
# --- Save ------------------------------------------------------------------
# Single save point for the notebook. Sets cannot be represented in CSV, so the
# segment set is exported as a pipe-separated string and the set column dropped.
dyad_years['cpv_segments_str'] = dyad_years['cpv_segments'].apply(
    lambda x: '|'.join(sorted(x))
)

dyad_years.drop(columns=['cpv_segments']).to_csv('data_analysis/dyad_years_2.csv', index=False)
buyer_div_seg.to_csv('data_analysis/buyer_div_seg.csv', index=False)
supplier_dom_seg.to_csv('data_analysis/supplier_dom_seg.csv', index=False)
data.to_csv('data_analysis/data_clean_final.csv', index=False)

print("=== Final variable inventory: dyad_years ===")
for col in dyad_years.columns:
    na = dyad_years[col].isna().sum()
    print(f"  {col}: {dyad_years[col].dtype}, {na} NA ({na/len(dyad_years)*100:.1f}%)")

=== Final variable inventory: dyad_years ===
  CAE_NAME_CANONICAL: str, 0 NA (0.0%)
  WIN_NAME_FINAL: str, 0 NA (0.0%)
  YEAR: int64, 0 NA (0.0%)
  n_contracts_ijt: int64, 0 NA (0.0%)
  cpv_segments: object, 0 NA (0.0%)
  cumulative_ties_ijt: int64, 0 NA (0.0%)
  ever_tied_ijt: int64, 0 NA (0.0%)
  prev_year: float64, 29888 NA (79.1%)
  years_since_last_ijt: float64, 29888 NA (79.1%)
  first_tie_ijt: int64, 0 NA (0.0%)
  ysl_filled_ijt: float64, 0 NA (0.0%)
  first_observed_year_buyer: int64, 0 NA (0.0%)
  n_suppliers_it: int64, 0 NA (0.0%)
  log_n_suppliers_it: float64, 0 NA (0.0%)
  n_buyers_jt: int64, 0 NA (0.0%)
  n_active_buyers_t: int64, 0 NA (0.0%)
  supplier_share_jt: float64, 0 NA (0.0%)
  n_other_suppliers_it: int64, 0 NA (0.0%)
  log_n_other_suppliers_it: float64, 0 NA (0.0%)
  n_other_buyers_jt: int64, 0 NA (0.0%)
  supplier_share_excl_i_jt: float64, 0 NA (0.0%)
  cpv_segments_str: str, 0 NA (0.0%)
  cae_type_ijt: str, 0 NA (0.0%)
  has_single_bid_ijt: float64, 2560 NA (6.8

### Conclusion: Variable preparation

The analysis population is restricted to German public authorities; EU
institutions and international organisations, which appear in the German
selection only because of their location, are excluded.

The data are then collapsed to buyer–supplier–year dyads, following the binary
logic: a dyad-year records that a relationship existed, not how many
awards were published. On this structure, variables on the history of a dyad,
on the position of each buyer and supplier in a given year, and on award
context are constructed. Buyer- and supplier-level controls additionally enter
in leave-one-out form, excluding the focal dyad from its own controls.

Three datasets are exported because the analysis operates on different levels.
`dyad_years.csv` is the dyad-year panel and feeds the regression, where the
estimation sample is built via opportunity sets. `buyer_div_seg.csv` and
`supplier_dom_seg.csv` hold the buyer-year and supplier-year measures
separately by CPV segment, since segment-specific concentration cannot be
recovered from the pooled panel once dyads spanning several segments are
collapsed.